In [1]:
import gymnasium as gym
from gymnasium.spaces import Box, Dict, MultiDiscrete

from pprint import pprint
from copy import deepcopy
from pathlib import Path
from dataclasses import dataclass
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.multiprocessing as mp
from torch import nn
from torch.distributions import Normal, Categorical

from industrial_inventory_env import (
    IndustrialInventoryEnv,
    generate_student_config,
    public_config_summary,
)

np.set_printoptions(suppress=True)

In [2]:
# Configuration
ROLL_NUMBER = "DA25M622"
SEED = 2026
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Generate student configuration
student_config = generate_student_config(ROLL_NUMBER)
config_summary = public_config_summary(student_config)

print("Assigned configuration generated successfully.")
for key, value in config_summary.items():
    print(f"{key}: {value}")

Using device: cpu
Assigned configuration generated successfully.
project_version: IITM-6002W-RL-Inventory-2026-v1
roll_number: DA25M622
variant_id: V022
config_fingerprint: 7d77fc79debf206a
demand_multiplier_profile: [1.0, 1.1, 0.9]
initial_inventory_profile: [110, 100, 90]
lead_time_delay_profile: [0.1, 0.0, 0.05]


In [3]:
# Test environment
env = IndustrialInventoryEnv(
    student_config=student_config,
    scenario_mode="random",
    domain_randomization=True,
)

observation, info = env.reset(seed=2026)

print("Action space:", env.action_space)
print("Observation space:", env.observation_space)
pprint(info)

Action space: MultiDiscrete([11 11 11])
Observation space: Dict('arrival_pipeline': Box(0, 10000, (3, 4), int32), 'capacity_utilisation': Box(0.0, 1.0, (1,), float32), 'day': Box(0, 50, (1,), int32), 'demand_history': Box(0, 10000, (7, 3), int32), 'inventory': Box(0, 1000, (3,), int32))
{'config_fingerprint': '7d77fc79debf206a',
 'episode_parameters': {'delay_probabilities': [0.08, 0.0, 0.05],
                        'demand_multipliers': [1.05, 1.05, 0.85],
                        'initial_inventory': [110, 100, 90],
                        'scenario_components': ['seasonal', 'trend'],
                        'scenario_details': {'seasonal_amplitudes': [0.06241472342761341,
                                                                     0.09569493618795924,
                                                                     0.07088119371488985],
                                             'trend_direction': 'up',
                                             'trend_total_change'

In [ ]:
def set_global_seed(seed):
    """Seed Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def flatten_observation(obs):
    """
    Flatten the Dict observation space into a single 1D tensor.
    
    The observation contains:
    - arrival_pipeline: (3, 4) int32
    - capacity_utilisation: (1,) float32
    - day: (1,) int32
    - demand_history: (7, 3) int32
    - inventory: (3,) int32
    """
    flattened = []
    
    # arrival_pipeline: (3, 4) -> 12 values
    flattened.append(np.array(obs['arrival_pipeline']).flatten())
    
    # capacity_utilisation: (1,) -> 1 value
    flattened.append(np.array(obs['capacity_utilisation']).flatten())
    
    # day: (1,) -> 1 value
    flattened.append(np.array(obs['day']).flatten())
    
    # demand_history: (7, 3) -> 21 values
    flattened.append(np.array(obs['demand_history']).flatten())
    
    # inventory: (3,) -> 3 values
    flattened.append(np.array(obs['inventory']).flatten())
    
    # Total: 12 + 1 + 1 + 21 + 3 = 38 features
    return np.concatenate(flattened).astype(np.float32)


def flatten_observation_torch(obs_tensor):
    """Flatten observation tensor from Dict format to 1D tensor."""
    # For batch processing, we need to handle each observation in the batch
    if isinstance(obs_tensor, dict):
        # Single observation
        flattened = []
        flattened.append(obs_tensor['arrival_pipeline'].float().flatten())
        flattened.append(obs_tensor['capacity_utilisation'].float().flatten())
        flattened.append(obs_tensor['day'].float().flatten())
        flattened.append(obs_tensor['demand_history'].float().flatten())
        flattened.append(obs_tensor['inventory'].float().flatten())
        return torch.cat(flattened)
    else:
        # Batch of observations - obs_tensor is a dict of tensors
        flattened = []
        flattened.append(obs_tensor['arrival_pipeline'].float().reshape(-1, 12))
        flattened.append(obs_tensor['capacity_utilisation'].float().reshape(-1, 1))
        flattened.append(obs_tensor['day'].float().reshape(-1, 1))
        flattened.append(obs_tensor['demand_history'].float().reshape(-1, 21))
        flattened.append(obs_tensor['inventory'].float().reshape(-1, 3))
        return torch.cat(flattened, dim=1)


def obs_to_tensor(obs, device=DEVICE):
    """Convert observation dict to tensor."""
    if isinstance(obs, dict):
        tensor_dict = {}
        for key, value in obs.items():
            if isinstance(value, np.ndarray):
                tensor_dict[key] = torch.as_tensor(value, dtype=torch.float32, device=device)
            else:
                tensor_dict[key] = torch.tensor(value, dtype=torch.float32, device=device)
        return tensor_dict
    return obs


class ActorNetwork(nn.Module):
    """Gaussian actor for continuous action space."""
    
    def __init__(self, observation_dim=38, hidden_dim=128, action_dim=3):
        super().__init__()
        self.observation_dim = observation_dim
        self.action_dim = action_dim
        
        self.body = nn.Sequential(
            nn.Linear(observation_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        # Mean head: output logits for each product's order quantity (0-10)
        self.mean_head = nn.Linear(hidden_dim, action_dim * 11)  # 3 products * 11 possible quantities
        self.log_std = nn.Parameter(torch.zeros(action_dim * 11))
    
    def forward(self, states):
        # states can be dict or flattened tensor
        if isinstance(states, dict):
            states = flatten_observation_torch(states)
        
        features = self.body(states)
        mean_logits = self.mean_head(features)
        
        # Reshape to (batch_size, action_dim, 11) for categorical distribution
        batch_size = mean_logits.shape[0]
        mean_logits = mean_logits.view(batch_size, self.action_dim, 11)
        
        # For each product, get probability distribution over quantities 0-10
        return mean_logits  # Return logits for Categorical


class CriticNetwork(nn.Module):
    """Critic V(s) network."""
    
    def __init__(self, observation_dim=38, hidden_dim=128):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(observation_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
    
    def forward(self, states):
        if isinstance(states, dict):
            states = flatten_observation_torch(states)
        return self.network(states).squeeze(-1)


def a3c_worker(
    worker_id,
    global_actor,
    global_critic,
    actor_optimizer,
    critic_optimizer,
    update_lock,
    result_queue,
    config,
    student_config,
):
    """A3C worker process."""
    worker_seed = config.seed + 1000 * worker_id
    set_global_seed(worker_seed)
    env = IndustrialInventoryEnv(
        student_config=student_config,
        scenario_mode="random",
        domain_randomization=True,
    )

    local_actor = ActorNetwork().to(DEVICE)
    local_critic = CriticNetwork().to(DEVICE)

    # Copy global parameters
    local_actor.load_state_dict(global_actor.state_dict())
    local_critic.load_state_dict(global_critic.state_dict())

    state, info = env.reset(seed=worker_seed)
    running_return = 0.0
    running_length = 0
    total_episodes = 0

    for local_update in range(config.updates_per_worker):
        states = []
        actions = []
        training_rewards = []
        log_probs_list = []

        completed_episodes = []
        terminal_end = False
        final_bootstrap_state = None
        episode_return = 0.0
        episode_length = 0

        for _ in range(config.rollout_steps):
            state_tensor = obs_to_tensor(state)
            
            with torch.no_grad():
                logits = local_actor(state_tensor)
                # logits shape: (1, 3, 11) for single state
                if logits.dim() == 3:
                    logits = logits.squeeze(0)  # (3, 11)
                
                # Sample actions for each product independently
                sampled_actions = []
                log_probs = []
                for product_idx in range(3):
                    dist = Categorical(logits=logits[product_idx])
                    action = dist.sample()
                    sampled_actions.append(action.item())
                    log_probs.append(dist.log_prob(action))
                
                action = np.array(sampled_actions, dtype=np.int64)

            next_state, reward, terminated, truncated, info = env.step(action)
            
            # Store for training
            states.append(state_tensor)
            actions.append(action)
            training_rewards.append(float(reward))
            log_probs_list.append(torch.stack(log_probs))
            
            running_return += float(reward)
            running_length += 1
            episode_return += float(reward)
            episode_length += 1
            
            state = next_state
            
            if terminated or truncated:
                completed_episodes.append({
                    "return": running_return,
                    "length": running_length,
                })
                running_return = 0.0
                running_length = 0
                terminal_end = bool(terminated)
                final_bootstrap_state = obs_to_tensor(next_state) if not terminal_end else None
                state, info = env.reset()
                break

        # Calculate returns
        if terminal_end:
            R = 0.0
        else:
            bootstrap_state = obs_to_tensor(state)
            with torch.no_grad():
                R = float(local_critic(bootstrap_state).item())

        returns = []
        for r in reversed(training_rewards):
            R = r + config.gamma * R
            returns.append(R)
        returns.reverse()

        # Prepare batch tensors
        # Stack states into batch
        state_batch = []
        for s in states:
            state_batch.append(flatten_observation_torch(s))
        state_batch = torch.stack(state_batch)  # (batch_size, 38)
        
        actions_batch = torch.tensor(actions, dtype=torch.long, device=DEVICE)  # (batch_size, 3)
        returns_batch = torch.as_tensor(returns, dtype=torch.float32, device=DEVICE)  # (batch_size,)

        # Compute values and advantages
        values_batch = local_critic(state_batch)  # (batch_size,)
        advantages_batch = returns_batch - values_batch

        # Compute actor loss
        logits_batch = local_actor(state_batch)  # (batch_size, 3, 11)
        actor_loss = 0.0
        entropy = 0.0
        
        for batch_idx in range(logits_batch.shape[0]):
            for product_idx in range(3):
                dist = Categorical(logits=logits_batch[batch_idx, product_idx])
                action = actions_batch[batch_idx, product_idx]
                log_prob = dist.log_prob(action)
                actor_loss += -log_prob * advantages_batch[batch_idx].detach()
                entropy += dist.entropy()
        
        actor_loss = actor_loss / (logits_batch.shape[0] * 3)
        entropy = entropy / (logits_batch.shape[0] * 3)
        actor_loss -= config.entropy_coefficient * entropy

        # Compute critic loss
        critic_loss = 0.5 * advantages_batch.pow(2).mean()

        # Backward
        local_actor.zero_grad()
        local_critic.zero_grad()
        actor_loss.backward()
        critic_loss.backward()

        # Update global models with lock
        with update_lock:
            for global_param, local_param in zip(global_actor.parameters(), local_actor.parameters()):
                if local_param.grad is not None:
                    global_param.grad = local_param.grad.clone()
            
            for global_param, local_param in zip(global_critic.parameters(), local_critic.parameters()):
                if local_param.grad is not None:
                    global_param.grad = local_param.grad.clone()
            
            torch.nn.utils.clip_grad_norm_(global_actor.parameters(), config.max_grad_norm)
            torch.nn.utils.clip_grad_norm_(global_critic.parameters(), config.max_grad_norm)
            
            actor_optimizer.step()
            critic_optimizer.step()

        # Refresh local networks
        local_actor.load_state_dict(global_actor.state_dict())
        local_critic.load_state_dict(global_critic.state_dict())

        # Send completed episodes
        for ep in completed_episodes:
            result_queue.put(ep)
            total_episodes += 1

    env.close()
    result_queue.put(None)  # Signal worker completion


@dataclass
class A3CConfig:
    num_workers: int = 4
    updates_per_worker: int = 200  # Total updates = num_workers * updates_per_worker
    rollout_steps: int = 20
    gamma: float = 0.99
    actor_learning_rate: float = 1e-4
    critic_learning_rate: float = 3e-4
    entropy_coefficient: float = 0.01
    max_grad_norm: float = 0.5
    seed: int = SEED


def train_a3c(config=A3CConfig(), student_config=student_config):
    """Train asynchronous advantage actor-critic."""
    set_global_seed(config.seed)

    global_actor = ActorNetwork().to(DEVICE)
    global_critic = CriticNetwork().to(DEVICE)

    # Share memory
    global_actor.share_memory()
    global_critic.share_memory()

    actor_optimizer = torch.optim.Adam(global_actor.parameters(), lr=config.actor_learning_rate)
    critic_optimizer = torch.optim.Adam(global_critic.parameters(), lr=config.critic_learning_rate)

    update_lock = mp.Lock()
    result_queue = mp.Queue()

    processes = []
    for worker_id in range(config.num_workers):
        p = mp.Process(
            target=a3c_worker,
            args=(
                worker_id,
                global_actor,
                global_critic,
                actor_optimizer,
                critic_optimizer,
                update_lock,
                result_queue,
                config,
                student_config,
            ),
        )
        p.start()
        processes.append(p)

    completed_returns = []
    completed_lengths = []
    active_workers = config.num_workers

    print("Starting A3C training...")
    
    while active_workers > 0:
        result = result_queue.get()
        if result is None:
            active_workers -= 1
            print(f"Worker finished. Active workers: {active_workers}")
        else:
            completed_returns.append(result["return"])
            completed_lengths.append(result["length"])

            if len(completed_returns) % 25 == 0:
                print(
                    f"Episodes: {len(completed_returns):4d} | "
                    f"mean return: {np.mean(completed_returns[-25:]):8.1f} | "
                    f"mean length: {np.mean(completed_lengths[-25:]):6.1f}"
                )

    for p in processes:
        p.join()

    history = {
        "episode_returns": np.asarray(completed_returns, dtype=np.float32),
        "episode_lengths": np.asarray(completed_lengths, dtype=np.int32),
    }
    return global_actor, global_critic, history


def evaluate(actor, num_episodes=30, seed=2026):
    """Evaluate the trained actor."""
    env = IndustrialInventoryEnv(
        student_config=student_config,
        scenario_mode="random",
        domain_randomization=True,
    )
    returns = np.zeros(num_episodes, dtype=np.float32)
    lengths = np.zeros(num_episodes, dtype=np.int32)
    actor.eval()

    for episode_index in range(num_episodes):
        episode_seed = seed + episode_index
        state, info = env.reset(seed=episode_seed)
        episode_return = 0.0
        episode_length = 0

        for step in range(50):  # Max episode length
            state_tensor = obs_to_tensor(state)
            with torch.no_grad():
                logits = actor(state_tensor)
                if logits.dim() == 3:
                    logits = logits.squeeze(0)
                
                # Greedy action selection (deterministic policy)
                actions = []
                for product_idx in range(3):
                    dist = Categorical(logits=logits[product_idx])
                    action = dist.probs.argmax()
                    actions.append(action.item())
                action = np.array(actions, dtype=np.int64)

            state, reward, terminated, truncated, info = env.step(action)
            episode_return += float(reward)
            episode_length += 1

            if terminated or truncated:
                break

        returns[episode_index] = episode_return
        lengths[episode_index] = episode_length
        
        if (episode_index + 1) % 10 == 0:
            print(f"Evaluation episode {episode_index + 1}/{num_episodes}: return = {episode_return:.1f}")

    env.close()
    actor.train()
    return {
        "returns": returns,
        "lengths": lengths,
    }

In [ ]:
# === Train A3C ===
print("\n" + "="*50)
print("Starting A3C Training")
print("="*50)

a3c_actor, a3c_critic, a3c_history = train_a3c()

print("\n" + "="*50)
print("A3C Training Complete")
print("="*50)
print(f"Total episodes: {len(a3c_history['episode_returns'])}")
print(f"Mean return: {a3c_history['episode_returns'].mean():.1f}")
print(f"Best return: {a3c_history['episode_returns'].max():.1f}")
print(f"Mean episode length: {a3c_history['episode_lengths'].mean():.1f}")

In [ ]:
# === Evaluate trained model ===
print("\n" + "="*50)
print("Evaluating A3C Model")
print("="*50)

a3c_eval = evaluate(a3c_actor, num_episodes=30)

print("\nEvaluation Results:")
print(f"Mean return: {a3c_eval['returns'].mean():.1f} +/- {a3c_eval['returns'].std():.1f}")
print(f"Median return: {np.median(a3c_eval['returns']):.1f}")
print(f"Min return: {a3c_eval['returns'].min():.1f}")
print(f"Max return: {a3c_eval['returns'].max():.1f}")
print(f"Mean episode length: {a3c_eval['lengths'].mean():.1f}")

# === Plot results ===
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Training returns
ax = axes[0, 0]
ax.plot(a3c_history['episode_returns'])
ax.set_title('Training Episode Returns')
ax.set_xlabel('Episode')
ax.set_ylabel('Return')

# Rolling average
ax = axes[0, 1]
window = 25
rolling_avg = np.convolve(a3c_history['episode_returns'], np.ones(window)/window, mode='valid')
ax.plot(rolling_avg)
ax.set_title(f'Training Returns (Rolling Avg {window})')
ax.set_xlabel('Episode')
ax.set_ylabel('Return')

# Evaluation returns
ax = axes[1, 0]
ax.bar(range(len(a3c_eval['returns'])), a3c_eval['returns'])
ax.axhline(y=a3c_eval['returns'].mean(), color='r', linestyle='-', label=f"Mean: {a3c_eval['returns'].mean():.1f}")
ax.set_title('Evaluation Episode Returns')
ax.set_xlabel('Episode')
ax.set_ylabel('Return')
ax.legend()

# Episode lengths
ax = axes[1, 1]
ax.plot(a3c_history['episode_lengths'])
ax.set_title('Training Episode Lengths')
ax.set_xlabel('Episode')
ax.set_ylabel('Length')

plt.tight_layout()
plt.show()

In [ ]:
# Save the model
torch.save(a3c_actor.state_dict(), "a3c_actor_industrial_inventory.pth")
torch.save(a3c_critic.state_dict(), "a3c_critic_industrial_inventory.pth")
print("\nModels saved to disk.")